In [1]:
import os
import cv2
import gc
import sys
import random
import warnings
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from torchvision import models

# Albumentations for "Heavy" Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Suppress the harmless scheduler warning
warnings.filterwarnings("ignore", category=UserWarning)

# ==========================================
# 1. CONFIGURATION
# ==========================================
class Config:
    SEED = 42
    IMG_SIZE = 384
    BATCH_SIZE = 16
    EPOCHS = 30
    LR = 1e-4
    WEIGHT_DECAY = 1e-3
    N_FOLDS = 5
    NUM_WORKERS = 2 # Lowered to avoid multiprocessing locks in some environments
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- PATHS ---
    BASE_PATH = "/kaggle/input/dataset/"
    if os.path.exists(os.path.join(BASE_PATH, "train_data_cleaned")):
        TRAIN_ROOT = os.path.join(BASE_PATH, "train_data_cleaned")
    else:
        TRAIN_ROOT = os.path.join(BASE_PATH, "train_data")

    TRAIN_IMG_DIR = os.path.join(TRAIN_ROOT, "images")
    TRAIN_MASK_DIR = os.path.join(TRAIN_ROOT, "masks")
    TRAIN_LABELS_CSV = os.path.join(TRAIN_ROOT, "train_labels.csv")

    TEST_ROOT = os.path.join(BASE_PATH, "test_data")
    TEST_IMG_DIR = os.path.join(TEST_ROOT, "images")
    TEST_MASK_DIR = os.path.join(TEST_ROOT, "masks")

    # --- ACCELERATOR PATHS ---
    FAST_TRAIN_IMG = "/kaggle/working/fast_train/images"
    FAST_TRAIN_MASK = "/kaggle/working/fast_train/masks"
    FAST_TEST_IMG = "/kaggle/working/fast_test/images"
    FAST_TEST_MASK = "/kaggle/working/fast_test/masks"

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(Config.SEED)

# ==========================================
# 2. UTILS: Reinhard & Smart Crop
# ==========================================
class ReinhardNormalizer:
    def __init__(self):
        self.target_mean = np.array([148.60, 169.30, 105.47])
        self.target_std = np.array([41.56, 11.23, 14.39])

    def __call__(self, img_rgb):
        try:
            img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype("float32")
            img_mean = np.mean(img_lab, axis=(0,1))
            img_std = np.std(img_lab, axis=(0,1)) + 1e-5
            img_lab = (img_lab - img_mean) / img_std * self.target_std + self.target_mean
            img_lab = np.clip(img_lab, 0, 255).astype("uint8")
            return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
        except:
            return img_rgb

def smart_square_crop_dual(img, mask):
    """
    Crops BOTH image and mask to the same square ROI based on the mask's content.
    """
    if np.sum(mask) == 0:
        h, w = img.shape[:2]
        sz = min(h, w)
        y = (h-sz)//2
        x = (w-sz)//2
        return img[y:y+sz, x:x+sz], mask[y:y+sz, x:x+sz]

    rows, cols = np.where(mask > 0)
    y_min, y_max = np.min(rows), np.max(rows)
    x_min, x_max = np.min(cols), np.max(cols)

    h_roi = y_max - y_min
    w_roi = x_max - x_min
    side = max(h_roi, w_roi)

    center_y = y_min + h_roi // 2
    center_x = x_min + w_roi // 2

    y_start = max(0, center_y - side // 2)
    y_end = min(img.shape[0], center_y + side // 2)
    x_start = max(0, center_x - side // 2)
    x_end = min(img.shape[1], center_x + side // 2)

    crop_img = img[y_start:y_end, x_start:x_end]
    crop_mask = mask[y_start:y_end, x_start:x_end]

    # Pad to square if needed
    pad_h = side - crop_img.shape[0]
    pad_w = side - crop_img.shape[1]

    if pad_h > 0 or pad_w > 0:
        crop_img = cv2.copyMakeBorder(
            crop_img,
            top=pad_h//2, bottom=pad_h - pad_h//2,
            left=pad_w//2, right=pad_w - pad_w//2,
            borderType=cv2.BORDER_CONSTANT, value=[255, 255, 255]
        )
        crop_mask = cv2.copyMakeBorder(
            crop_mask,
            top=pad_h//2, bottom=pad_h - pad_h//2,
            left=pad_w//2, right=pad_w - pad_w//2,
            borderType=cv2.BORDER_CONSTANT, value=0
        )

    return crop_img, crop_mask

# ==========================================
# 3. ACCELERATOR: PRE-COMPUTE & SAVE
# ==========================================
def preprocess_dual_and_save(df, img_dir, mask_dir, out_img_dir, out_mask_dir, is_test=False):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)

    normalizer = ReinhardNormalizer()
    valid_files = []

    print(f"⚡ Optimizing data to {out_img_dir}...")

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # --- ROBUST FILENAME EXTRACTION ---
        if is_test:
            filename = row['filename']
        else:
            if 'image_name' in row: filename = row['image_name']
            elif 'sample_index' in row: filename = row['sample_index']
            elif 'filename' in row: filename = row['filename']
            else: filename = row[0]

        # Mask naming fix
        if "img_" in filename: mask_name = filename.replace("img_", "mask_")
        elif "img" in filename: mask_name = filename.replace("img", "mask")
        else: mask_name = "mask_" + filename

        img_p = os.path.join(img_dir, filename)
        mask_p = os.path.join(mask_dir, mask_name)
        if not os.path.exists(mask_p): mask_p = mask_p.replace(".jpg", ".png").replace(".jpeg", ".png")

        save_img_p = os.path.join(out_img_dir, filename.split('.')[0]+'.png')
        save_mask_p = os.path.join(out_mask_dir, filename.split('.')[0]+'.png')

        # Skip if already done
        if os.path.exists(save_img_p) and os.path.exists(save_mask_p):
            valid_files.append(filename.split('.')[0]+'.png')
            continue

        try:
            # Load
            img = cv2.imread(img_p)
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE)

            # Smart Crop
            if mask is not None:
                img, mask = smart_square_crop_dual(img, mask)
            else:
                img = cv2.resize(img, (Config.IMG_SIZE, Config.IMG_SIZE))
                mask = np.zeros((Config.IMG_SIZE, Config.IMG_SIZE), dtype=np.uint8)

            # Reinhard
            img = normalizer(img)

            # Resize Final
            img = cv2.resize(img, (Config.IMG_SIZE, Config.IMG_SIZE), interpolation=cv2.INTER_LANCZOS4)
            mask = cv2.resize(mask, (Config.IMG_SIZE, Config.IMG_SIZE), interpolation=cv2.INTER_NEAREST)

            # Save
            Image.fromarray(img).save(save_img_p)
            Image.fromarray(mask).save(save_mask_p)
            valid_files.append(filename.split('.')[0]+'.png')

        except Exception as e:
            continue

    return valid_files

# --- EXECUTE PREPROCESSING ---
df_train_raw = pd.read_csv(Config.TRAIN_LABELS_CSV)
if 'image_name' not in df_train_raw.columns:
    df_train_raw.rename(columns={'sample_index': 'image_name', 'filename': 'image_name'}, inplace=True)

le = LabelEncoder()
df_train_raw['label_encoded'] = le.fit_transform(df_train_raw['label'])
NUM_CLASSES = len(le.classes_)
print(f"Classes: {le.classes_}")

# Run for Train
valid_train_files = preprocess_dual_and_save(
    df_train_raw, Config.TRAIN_IMG_DIR, Config.TRAIN_MASK_DIR, Config.FAST_TRAIN_IMG, Config.FAST_TRAIN_MASK, is_test=False
)
df_train_raw['processed_filename'] = df_train_raw['image_name'].apply(lambda x: x.split('.')[0]+'.png')
df_train = df_train_raw[df_train_raw['processed_filename'].isin(valid_train_files)].reset_index(drop=True)
print(f"Valid Train Images: {len(df_train)}")

# Run for Test
test_files = os.listdir(Config.TEST_IMG_DIR)
df_test_raw = pd.DataFrame({'filename': test_files})
valid_test_files = preprocess_dual_and_save(
    df_test_raw, Config.TEST_IMG_DIR, Config.TEST_MASK_DIR, Config.FAST_TEST_IMG, Config.FAST_TEST_MASK, is_test=True
)
df_test_raw['processed_filename'] = df_test_raw['filename'].apply(lambda x: x.split('.')[0]+'.png')
df_test = df_test_raw[df_test_raw['processed_filename'].isin(valid_test_files)].reset_index(drop=True)
print(f"Valid Test Images: {len(df_test)}")

# ==========================================
# 4. DATASET & AUGMENTATIONS
# ==========================================
class DualStreamDataset(Dataset):
    def __init__(self, df, img_dir, mask_dir, transforms=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['processed_filename']

        img_path = os.path.join(self.img_dir, fname)
        mask_path = os.path.join(self.mask_dir, fname)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        mask = mask.float() / 255.0

        if self.is_test:
            # Handle possible naming variations for submission
            orig_name = row['filename'] if self.is_test else row['image_name']
            return image, mask, orig_name

        return image, mask, torch.tensor(row['label_encoded'], dtype=torch.long)

def get_transforms(data='train'):
    if data == 'train':
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5),
            # Color jitter only on RGB
            A.OneOf([
                A.HueSaturationValue(p=1.0),
                A.RandomBrightnessContrast(p=1.0),
            ], p=0.7),
            A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

# ==========================================
# 5. MODEL: DUAL-STREAM TEMPLE
# ==========================================
class DualStreamModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # --- Stream 1: RGB (Texture) ---
        self.rgb_backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        rgb_dim = self.rgb_backbone.classifier[1].in_features
        self.rgb_backbone.classifier = nn.Identity()

        # --- Stream 2: Mask (Geometry) ---
        self.mask_backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        # Modify first layer: 3 channels -> 1 channel
        orig_conv = self.mask_backbone.features[0][0]
        new_conv = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        with torch.no_grad():
            new_conv.weight[:] = torch.mean(orig_conv.weight, dim=1, keepdim=True)
        self.mask_backbone.features[0][0] = new_conv
        mask_dim = self.mask_backbone.classifier[1].in_features
        self.mask_backbone.classifier = nn.Identity()

        # --- The Summit (Fusion) ---
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Sequential(
            nn.Linear(rgb_dim + mask_dim, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, rgb, mask):
        feat_rgb = self.rgb_backbone(rgb)
        feat_mask = self.mask_backbone(mask)
        combined = torch.cat((feat_rgb, feat_mask), dim=1)
        combined = self.dropout(combined)
        return self.fc(combined)

# ==========================================
# 6. TRAINING ENGINE
# ==========================================
def mixup_dual(rgb, mask, y, alpha=1.0):
    if alpha > 0: lam = np.random.beta(alpha, alpha)
    else: lam = 1
    batch_size = rgb.size(0)
    index = torch.randperm(batch_size).to(Config.DEVICE)
    mixed_rgb = lam * rgb + (1 - lam) * rgb[index, :]
    mixed_mask = lam * mask + (1 - lam) * mask[index, :]
    y_a, y_b = y, y[index]
    return mixed_rgb, mixed_mask, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_fold(fold, train_idx, val_idx):
    print(f"\n================ FOLD {fold+1}/{Config.N_FOLDS} ================")

    train_data = df_train.iloc[train_idx].reset_index(drop=True)
    val_data = df_train.iloc[val_idx].reset_index(drop=True)

    # Weighted Sampler
    class_counts = train_data['label_encoded'].value_counts().sort_index().values
    class_weights = 1. / class_counts
    sample_weights = [class_weights[l] for l in train_data['label_encoded']]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    train_ds = DualStreamDataset(train_data, Config.FAST_TRAIN_IMG, Config.FAST_TRAIN_MASK, transforms=get_transforms('train'))
    val_ds = DualStreamDataset(val_data, Config.FAST_TRAIN_IMG, Config.FAST_TRAIN_MASK, transforms=get_transforms('val'))

    # *** FIX: Added drop_last=True to prevent BatchNorm error with single item batch ***
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=Config.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True)

    model = DualStreamModel(NUM_CLASSES).to(Config.DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)

    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=1e-3, steps_per_epoch=len(train_loader), epochs=Config.EPOCHS, pct_start=0.3
    )
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()

    best_f1 = 0.0
    patience = 6
    counter = 0

    for epoch in range(Config.EPOCHS):
        model.train()
        train_loss = 0

        for rgb, mask, labels in train_loader:
            rgb = rgb.to(Config.DEVICE)
            mask = mask.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            optimizer.zero_grad()
            with autocast():
                if np.random.random() > 0.5:
                    rgb, mask, ta, tb, lam = mixup_dual(rgb, mask, labels, 0.4)
                    out = model(rgb, mask)
                    loss = mixup_criterion(criterion, out, ta, tb, lam)
                else:
                    out = model(rgb, mask)
                    loss = criterion(out, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Unconditional step to avoid PyTorch version conflicts
            scheduler.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        preds_all, targets_all = [], []
        with torch.no_grad():
            for rgb, mask, labels in val_loader:
                rgb, mask = rgb.to(Config.DEVICE), mask.to(Config.DEVICE)
                out = model(rgb, mask)
                preds = torch.argmax(out, dim=1).cpu().numpy()
                preds_all.extend(preds)
                targets_all.extend(labels.numpy())

        val_f1 = f1_score(targets_all, preds_all, average='macro')
        print(f"Ep {epoch+1:02d} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Best: {best_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f"dual_fold_{fold}.pth")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return best_f1

# --- RUN TRAIN ---
skf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
fold_scores = []
for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train['label_encoded'])):
    score = train_fold(fold, train_idx, val_idx)
    fold_scores.append(score)

print(f"\n==================================")
print(f"FINAL CV F1: {np.mean(fold_scores):.4f}")
print(f"==================================")

# ==========================================
# 7. INFERENCE (Dual TTA)
# ==========================================
print("\nStarting Inference...")
test_ds = DualStreamDataset(df_test, Config.FAST_TEST_IMG, Config.FAST_TEST_MASK, transforms=get_transforms('val'), is_test=True)
test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

models_list = []
for i in range(Config.N_FOLDS):
    m = DualStreamModel(NUM_CLASSES).to(Config.DEVICE)
    m.load_state_dict(torch.load(f"dual_fold_{i}.pth"))
    m.eval()
    models_list.append(m)

final_preds, final_fnames = [], []

with torch.no_grad():
    for rgb, mask, fnames in tqdm(test_loader):
        rgb = rgb.to(Config.DEVICE)
        mask = mask.to(Config.DEVICE)

        # Dual TTA
        rgb_inputs = [rgb, torch.flip(rgb, [3]), torch.flip(rgb, [2]), torch.rot90(rgb, 1, [2, 3])]
        mask_inputs = [mask, torch.flip(mask, [3]), torch.flip(mask, [2]), torch.rot90(mask, 1, [2, 3])]

        avg_probs = torch.zeros((rgb.size(0), NUM_CLASSES)).to(Config.DEVICE)

        for model in models_list:
            for i in range(len(rgb_inputs)):
                out = model(rgb_inputs[i], mask_inputs[i])
                avg_probs += torch.softmax(out, dim=1)

        avg_probs /= (len(models_list) * len(rgb_inputs))

        preds = torch.argmax(avg_probs, dim=1).cpu().numpy()
        decoded = le.inverse_transform(preds)

        final_preds.extend(decoded)
        final_fnames.extend(fnames)

sub = pd.DataFrame({'sample_index': final_fnames, 'label': final_preds})
sub.to_csv("submission.csv", index=False)
print("Submission Saved!")

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Number of training samples: 581
Classes: ['HER2(+)', 'Luminal A', 'Luminal B', 'Triple negative']


Preloading train images:   6%|▌         | 34/581 [00:09<02:35,  3.53it/s]


KeyboardInterrupt: 